### Chapter 7.1 - From Fully Connected Layers to Convolutions

Convolutional networks are neural networks that use the spatial structure of images instead of treating every pixel as an unrelated feature.


In [ ]:
import torch
from torch import nn


### 7.1.1 Invariance

#### 1. Intuition

Invariance means a model's answer should stay the same when an irrelevant change happens.

For images, the exact location of a useful pattern often changes. A vertical edge may appear on the left, center, or right, but it is still a vertical edge.

#### 2. Why this exists

Fully connected layers learn separate weights for every input position. That makes them inefficient when the same pattern can appear at many positions.


#### 3. Examples

A tiny detector can be reused at several positions.


In [ ]:
image = [0, 1, 1, 0, 1]
pattern = [1, 1]
hits = []
for i in range(len(image) - 1):
    hits.append(image[i:i + 2] == pattern)
hits


#### 4. Step-by-step breakdown

`image` is a one-dimensional toy image.

`pattern` is the local feature we want to detect.

The loop moves the same pattern detector across positions.

The result records where the pattern appears.

#### 5. Connection to ML systems

CNNs use this idea in two dimensions: the same small filter is applied across many image locations.

#### 6. Common confusion points

- Invariance does not mean ignoring all location information.
- It means small shifts should not force learning a brand-new detector.
- A detector can be local even when the final prediction is global.
- Pooling and later layers help turn local detections into stable predictions.


### 7.1.2 Constraining the MLP

#### 1. Intuition

Constraining an MLP means deliberately limiting which weights it can use.

Instead of connecting every pixel to every hidden unit, we can force each hidden unit to look only at a small neighborhood and reuse the same weights everywhere.

#### 2. Why this exists

Images have local structure. A nearby pixel usually matters more than a far-away pixel for detecting edges, corners, and textures.


#### 3. Examples

Compare parameter counts for a tiny image.


In [ ]:
height, width = 4, 4
hidden_units = 3
mlp_params = height * width * hidden_units
kernel_params = 2 * 2
mlp_params, kernel_params


The local kernel uses far fewer weights.


In [ ]:
saving = mlp_params - kernel_params
saving


#### 4. Step-by-step breakdown

A 4 by 4 image has 16 pixel positions.

A fully connected hidden layer with 3 units needs 48 weights.

A 2 by 2 reusable kernel needs only 4 weights.

The same kernel can scan many local neighborhoods.

#### 5. Connection to ML systems

Convolutional layers are constrained linear layers with local connectivity and weight sharing.

#### 6. Common confusion points

- Constraints are not weakness by default.
- Good constraints encode useful assumptions.
- CNNs assume local patterns matter and can repeat across space.
- Bad constraints can hurt when the assumption is false.


### 7.1.3 Convolutions

#### 1. Intuition

A convolutional layer applies a small matrix of weights, called a kernel or filter, over local windows of an image.

In deep learning libraries, the operation is usually cross-correlation: the kernel is not flipped before sliding.

#### 2. Why this exists

Convolutions let the model learn local feature detectors while sharing the same detector across the whole image.


#### 3. Examples

Compute one local response by hand.


In [ ]:
window = [[1, 2], [3, 4]]
kernel = [[1, 0], [0, -1]]
score = 0
for r in range(2):
    for c in range(2):
        score += window[r][c] * kernel[r][c]
score


#### 4. Step-by-step breakdown

`window` is the local image patch.

`kernel` contains learned weights.

Each aligned pair is multiplied.

The products are added into one output value.

#### 5. Connection to ML systems

PyTorch's `nn.Conv2d` performs this sliding local weighted sum across batches, channels, height, and width.

#### 6. Common confusion points

- Deep learning often says convolution while computing cross-correlation.
- The kernel values are learned during training.
- One output number summarizes one local window.
- The same kernel is reused at many positions.


### 7.1.4 Channels

#### 1. Intuition

A channel is a separate measurement at each spatial location.

A grayscale image has one channel. A color image commonly has red, green, and blue channels. Hidden CNN layers also have channels, but those channels represent learned feature maps.

#### 2. Why this exists

Channels let a model store multiple kinds of evidence at the same spatial location.


#### 3. Examples

Create a tiny color-like image tensor.


In [ ]:
X = torch.zeros(3, 2, 2)
X[0] += 1
X[1] += 2
X[2] += 3
X.shape, X[:, 0, 0]


#### 4. Step-by-step breakdown

The tensor shape is channels, height, width.

Channel 0 could represent red-like values.

Channel 1 and 2 store different measurements.

At location `(0, 0)`, three channel values are available.

#### 5. Connection to ML systems

CNNs treat channels as feature dimensions attached to each pixel or hidden spatial location.

#### 6. Common confusion points

- Channels are not extra image rows.
- A convolution kernel usually spans all input channels.
- Output channels are learned feature maps.
- Shape order differs across libraries, so always inspect it.


### 7.1.5 Summary and Discussion

#### 1. Intuition

CNNs combine local connectivity, weight sharing, and channels.

These choices match image structure better than flattening every pixel immediately.

#### 2. Why this exists

The design makes image models more parameter-efficient and better at reusing local feature detectors.


#### 3. Examples

Represent the three CNN assumptions as plain strings.


In [ ]:
assumptions = [
    "nearby pixels are related",
    "useful patterns repeat across positions",
    "channels store multiple measurements",
]
assumptions


#### 4. Step-by-step breakdown

The first assumption motivates local windows.

The second motivates shared kernels.

The third motivates multiple feature maps.

Together they define the core CNN bias.

#### 5. Connection to ML systems

Modern vision models still rely heavily on these ideas, even when they use much deeper architectures.

#### 6. Common confusion points

- CNNs are not only for photographs.
- Any grid-like data may benefit from convolution.
- Flattening too early discards spatial organization.
- Later dense layers can still use high-level CNN features.


### 7.1.6 Exercises

#### 1. Intuition

These exercises check the basic ideas behind convolutions.

#### 2. Why this exists

Before writing larger CNNs, you should be able to reason about locality, sharing, and channels with tiny examples.


#### 3. Examples

Exercise 1: count values in a 3-channel 2 by 2 image.


In [ ]:
channels, height, width = 3, 2, 2
total_values = channels * height * width
total_values


Exercise 2: count weights in two 3 by 3 kernels.


In [ ]:
input_channels = 3
output_channels = 2
weights = output_channels * input_channels * 3 * 3
weights


#### 4. Step-by-step breakdown

Exercise 1 multiplies channels by spatial size.

Exercise 2 multiplies output channels, input channels, and kernel area.

This is the starting point for understanding CNN parameter counts.

#### 5. Connection to ML systems

Model summaries report convolutional parameter counts using this same shape logic.

#### 6. Common confusion points

- Output channels mean number of learned kernels.
- Input channels are consumed by each kernel.
- Kernel height and width define local window size.
- Bias terms add one extra value per output channel if enabled.
